<a href="https://colab.research.google.com/github/YokoyamaLab/PythonBasics/blob/2025/27_day08tb_YoloDetect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**ノート27**

# [Day08 Textbook] 画像物体認識の試行

## 🌀YOLOとは


YOLO（You Only Look Once）は、物体検出のための高速かつ高精度なアルゴリズムです。YOLOは、画像や動画内の物体をリアルタイムで検出する能力を持ち、特に自動運転やセキュリティカメラなどの用途で広く利用されています。

YOLOの特徴：
* **高速処理**：YOLOは画像全体を一度に解析し、物体を検出するため、従来の手法よりも高速です。
* **高精度**：一度の視認で複数の物体を同時に検出し、識別することができます。
* **汎用性**：様々な形状やサイズの画像に対して安定した検出性能を発揮します。

YOLOのデメリット：

* **小さな物体の検出が難しい**：物体がセルの境界にまたがる場合、検出精度が低下することがあります。
* **複雑な背景での誤検出**：背景が複雑な場合、誤検出や検出漏れが発生する可能性があります。

それではまず必要なライブラリをインストールします。

## 🌀YOLOの事前準備

In [ ]:
# 必要なライブラリのインストール
!pip install ultralytics
!pip install ipywidgets
!pip install sqids

# ライブラリのインポート
import cv2
import numpy as np
from google.colab.patches import cv2_imshow
import ipywidgets as widgets
from IPython.display import display
from ultralytics import YOLO
from IPython.display import clear_output
from PIL import Image, ImageDraw, ImageFont
from sqids import Sqids
import datetime
import sys
import shutil
import os

# 画像保存のためのコード
from google.colab import drive
drive.mount("/content/gdrive")
our_dir = "/content/gdrive/Shareddrives/2025-LG080A01／情報科学 c/yolo/"

def copy_file_to_our_dir(origin_path,destination_filename=""):
  print(f"'{origin_path}' を '{our_dir}' にコピーします。")
  if not os.path.exists(our_dir):
      os.makedirs(our_dir)
  shutil.copy2(origin_path, our_dir + destination_filename)


## 🌀物体認識の実行

次にYOLOで物体認識を行うコードを記します。実行すると下にUploadボタンが出てきますので、クリックして何か画像を選んでください。認識したものが画像中に矩形でくくられて表示されます。一緒に表示されている数値は確信度といって、どれくらい自信をもってその物体だと認識しているかの値です。

In [ ]:
# 学籍番号
No = "G00000"

# 実行事にユニークなファイル名を生成する
dt = datetime.datetime.now()
sqids = Sqids(min_length=10)
current_execution = sqids.encode([dt.hour, dt.minute, dt.second])
filename_original = f"{No}-{current_execution}-original.jpg"
filename_annotated = f"{No}-{current_execution}-annotated.jpg"


# YOLOv11モデルのロード
# 'yolov11x.pt' は特大モデル、他のモデル（s, m, l, x）も選択可能
model = YOLO('yolo11x.pt')

# ファイルアップロードウィジェットの作成
uploader = widgets.FileUpload(
    accept='.jpg,.jpeg',  # 受け付けるファイルの種類
    multiple=False  # 単一ファイルのみ
)

output = widgets.Output()

def on_upload_change(change):
    with output:
        output.clear_output()
        if uploader.value:
            uploaded_file_name = list(uploader.value.keys())[0]
            uploaded_file_content = uploader.value[uploaded_file_name]['content']

            # アップロードされたファイルを一時的に保存
            with open(filename_original, 'wb') as f:
                f.write(uploaded_file_content)
            print(f"ファイルをアップロードしました: {uploaded_file_name}->{filename_original}")

            try:
                # YOLOによる物体認識の実行
                # save=Trueで結果画像をruns/detect/exp*/ に保存
                results = model(filename_original, save=True)

                # 結果が保存されたパスを取得
                # 通常、runs/detect/predictN/uploaded_file_name の形式で保存されます
                # runs/detect ディレクトリ内の最新のディレクトリを取得
                runs_dir = 'runs/detect'
                if os.path.exists(runs_dir):
                    exp_dirs = [os.path.join(runs_dir, d) for d in os.listdir(runs_dir) if os.path.isdir(os.path.join(runs_dir, d))]
                    exp_dirs.sort(key=os.path.getmtime, reverse=True) # 最新のディレクトリを先頭に
                    if exp_dirs:
                        result_image_path = os.path.join(exp_dirs[0], filename_original)

                        # 結果画像の表示
                        print("物体認識結果:"+result_image_path)
                        #img = Image(filename=result_image_path)
                        #display(img)
                        img = cv2.imread(result_image_path)
                        cv2_imshow(img)

                        # 結果ファイルをホームディレクトリにコピー
                        shutil.copy2(result_image_path, "./"+filename_annotated)

                        # 生成された一時ファイルを削除 (オプション)
                        # os.remove(uploaded_file_name)
                        # os.remove(result_image_path) # 結果画像を削除したい場合
                    else:
                        print("結果ディレクトリが見つかりませんでした。")
                else:
                    print("runs/detect ディレクトリが見つかりませんでした。")

            except Exception as e:
                print(f"物体認識中にエラーが発生しました: {e}")
                if os.path.exists(uploaded_file_name):
                     os.remove(uploaded_file_name)


# ファイルアップロードウィジェットの変更を監視
uploader.observe(on_upload_change, names='value')

# ウィジェットの表示
print("JPGファイルをアップロードしてください:")
display(uploader, output)


## 🌀物体認識結果の提出

何か面白い結果が得られたら、以下のコードを実行し提出用ディレクトリに画像をコピーしてください。

In [ ]:
# 結果のコピー
copy_file_to_our_dir(filename_original)
copy_file_to_our_dir(filename_annotated)